## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [3]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### masakhane/masakhaner2

In [6]:
label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-DATE": 7,
    "I-DATE": 8,
}

masakhaner2 = ner.ReadNERData()
masakhaner2_words, masakhaner2_labels = masakhaner2.read_dataset('masakhane/masakhaner2', label_map, lang='hau')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/5716 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/816 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1633 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/1633 [00:00<?, ?it/s]

In [7]:
print(ner.check_labels(masakhaner2_labels))
label_alignment = {
    'I-PER': 'I-PER',
    'I-DATE': 'O',
    'B-ORG': 'B-ORG',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'O':     'O',
    'B-DATE': 'O',
    'B-PER': 'B-PER',
    'I-ORG': 'I-ORG',
}

# Align the dataset labels to the standard labels
masakhaner2_labels = ner.align_dataset(masakhaner2_labels, label_alignment)
print(ner.check_labels(masakhaner2_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-ORG', 'B-LOC', 'I-DATE', 'I-LOC', 'B-PER', 'O', 'B-DATE', 'I-ORG', 'I-PER'}
{'B-ORG', 'B-LOC', 'I-LOC', 'B-PER', 'O', 'I-ORG', 'I-PER'}


# Evaluate model

In [8]:
alignment = {
'B-organization': 'B-ORG',
'O': 'O',
'B-other': 'O',
'B-person': 'B-PER',
'I-person': 'I-PER',
'B-location': 'B-LOC',
'I-organization': 'I-ORG',
'I-other': 'O',
'I-location': 'I-LOC'
}

model_name = "tner/xlm-roberta-large-conll2003"
model_name_output = 'tner-xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [9]:
# model_evaluation.model.config.id2label

### masakhane/masakhaner2

In [10]:
data_name = "masakhane/masakhaner2"
masakhaner2_evaluation_output = model_evaluation.evaluate_model(masakhaner2_words, masakhaner2_labels)

  0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
masakhaner2_seqeval = masakhaner2_evaluation_output.get_classification('Seqeval')
masakhaner2_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.4829,0.4526,0.4673,970
1,ORG,0.7490,0.7660,0.7574,1188
2,PER,0.8654,0.9481,0.9049,1194
3,micro,0.7229,0.7402,0.7314,3352
4,macro,0.6991,0.7222,0.7098,3352
5,weighted,0.7135,0.7402,0.7260,3352


In [12]:
masakhaner2_sklearn = masakhaner2_evaluation_output.get_classification('Sklearn')
masakhaner2_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.4978,0.4649,0.4808,970
1,B-ORG,0.7672,0.7776,0.7724,1187
2,B-PER,0.8833,0.9573,0.9188,1194
3,I-LOC,0.9529,0.1266,0.2234,640
4,I-ORG,0.8933,0.5330,0.6676,895
5,I-PER,0.9524,0.9657,0.9590,933
6,O,0.9721,0.9933,0.9825,39029
7,accuracy,0.9530,44848,None,None
8,macro,0.8456,0.6883,0.7149,44848
9,weighted,0.9518,0.9530,0.9468,44848
